# Workshop Pipeline — Clean Structured Data, Summarize Notes, Flag Smoking

This notebook builds on the starter harness pattern and applies it to the workshop dataset. It does three things:

1. **Cleans the structured CSV** (`patients_raw.csv`) — date formats, length-of-stay derivation, categorical normalization, lab-test consolidation, unit conversion, missing-value tokens, outliers, duplicates.
2. **Summarizes each patient's notes** using Ollama on HPC — one structured summary per patient from the `.txt` chart.
3. **Extracts a smoking flag** from the notes — independent signal that may or may not agree with the CSV `smoking_status` field.

Data is assumed to live at `/data/patients_raw.csv` and `/data/patient_notes/PT*.txt`. Adjust the paths in the Setup cell if your data lives elsewhere.

> All data and notes are synthetic. No real patients.

## 1. Workshop task and pod notes

Fill these in before you start coding.

- **Task title:**
- **Task description:**
- **Input type:** structured CSV + corpus of plain-text patient chart files
- **Desired output:** cleaned dataset joined with note summaries and an LLM-extracted smoking flag
- **One likely failure mode:**
- **Human review rule:**

## 2. Setup

Typical HPC shell setup outside the notebook:

```bash
ml ollama
ollama serve &
ollama list
ollama pull llama3.1:8b
```

If needed, install Python packages in this environment:

```bash
pip install ollama pandas
```

In [ ]:
from pathlib import Path
from datetime import datetime, date
import json
import re

import pandas as pd

# Paths — assumes data is staged at /data on the HPC. Adjust if needed.
DATA_DIR = Path('/data')
CSV_PATH = DATA_DIR / 'patients_raw.csv'
NOTES_DIR = DATA_DIR / 'patient_notes'
OUTPUT_DIR = Path('workshop_outputs')
OUTPUT_DIR.mkdir(exist_ok=True)

OLLAMA_MODEL = 'llama3.1:8b'
PROMPT_VERSION = 'v1'

# Cap the number of patients sent through the LLM at first.
# Bump up once your single-record test looks reasonable.
LLM_SUBSET_N = 20

print('CSV exists:    ', CSV_PATH.exists())
n_notes = sum(1 for _ in NOTES_DIR.glob('*.txt')) if NOTES_DIR.exists() else 0
print('Notes dir:     ', NOTES_DIR, '|', n_notes, 'files')
print('Output dir:    ', OUTPUT_DIR.resolve())

In [ ]:
# Optional: quick Ollama connectivity check
try:
    import ollama
    tags = ollama.list()
    print('Ollama reachable. Available models:')
    for m in tags.get('models', []):
        print(' -', m.get('model'))
except Exception as e:
    print('Could not reach Ollama. Start it from the terminal with `ollama serve &`.')
    print('Error:', e)

## 3. Part A — Clean the structured data

The raw CSV contains intentional data quality issues:

- 5 date formats mixed on admit/discharge
- length-of-stay sometimes pre-computed, sometimes derivable, sometimes missing
- categorical inconsistency: `sex`, `race`, `insurance`, `smoking_status`
- multiple labels per canonical lab test (e.g., `HbA1c` / `A1C` / `Hemoglobin A1c`)
- mixed units for weight (kg vs lbs) and some labs (mmol/L vs mg/dL)
- blood pressure mostly `120/80` but occasionally `120 over 80`
- heterogeneous missing tokens (`""`, `NA`, `?`, `-`, `unknown`, `null`)
- a few absurd age outliers
- 5 injected duplicate rows

Below we load and inspect, then apply per-column cleaners.

In [ ]:
df_raw = pd.read_csv(CSV_PATH, dtype=str, keep_default_na=False, na_values=[])
print(f'Loaded {len(df_raw)} rows (incl. duplicates).')
print('Columns:', list(df_raw.columns))
df_raw.head(3)

In [ ]:
# --- Canonicalization mappings ---

MISSING_TOKENS = {'', 'na', 'n/a', 'null', '?', '-', 'unknown'}

SEX_MAP = {
    'm': 'Male', 'male': 'Male',
    'f': 'Female', 'female': 'Female',
}

RACE_MAP = {
    'white': 'White', 'caucasian': 'White',
    'black': 'Black', 'african american': 'Black', 'african-american': 'Black', 'aa': 'Black',
    'asian': 'Asian',
    'hispanic': 'Hispanic', 'latino': 'Hispanic', 'latina': 'Hispanic',
    'native american': 'Native American', 'ai/an': 'Native American', 'american indian': 'Native American',
    'other': 'Other',
}

INSURANCE_MAP = {
    'medicare': 'Medicare',
    'medicaid': 'Medicaid',
    'private': 'Private',
    'self-pay': 'Self-Pay',
    'other': 'Other',
}

SMOKING_MAP = {
    'never': 'Never', 'n': 'Never',
    'former': 'Former',
    'current': 'Current', 'y': 'Current',
}

# Lab tests — note that 'na' here means Sodium, NOT missing.
# We apply this BEFORE coercing 'na' as missing for other columns.
LAB_TEST_MAP = {
    'hba1c': 'HbA1c', 'a1c': 'HbA1c', 'hemoglobin a1c': 'HbA1c',
    'hgba1c': 'HbA1c', 'glycated hemoglobin': 'HbA1c',
    'glucose': 'Glucose', 'glu': 'Glucose', 'blood glucose': 'Glucose',
    'serum glucose': 'Glucose', 'bg': 'Glucose',
    'creatinine': 'Creatinine', 'creat': 'Creatinine', 'cr': 'Creatinine',
    'serum creatinine': 'Creatinine',
    'ldl': 'LDL', 'ldl-c': 'LDL', 'ldl cholesterol': 'LDL',
    'low-density lipoprotein': 'LDL',
    'sodium': 'Sodium', 'na': 'Sodium', 'serum sodium': 'Sodium',
}

DATE_FORMATS = ['%Y-%m-%d', '%m/%d/%Y', '%d-%b-%Y', '%m-%d-%Y', '%Y/%m/%d']


def convert_lab(test, value, unit):
    """Convert lab value to canonical unit for the test, if a known alt unit."""
    if value is None or pd.isna(value) or test is None:
        return value, unit
    try:
        v = float(value)
    except (TypeError, ValueError):
        return None, unit
    if test == 'Glucose' and unit == 'mmol/L':
        return round(v * 18.0, 2), 'mg/dL'
    if test == 'Creatinine' and unit == 'umol/L':
        return round(v / 88.4, 2), 'mg/dL'
    if test == 'LDL' and unit == 'mmol/L':
        return round(v * 38.67, 2), 'mg/dL'
    return v, unit

In [ ]:
def is_missing(v):
    if v is None or (isinstance(v, float) and pd.isna(v)):
        return True
    return str(v).strip().lower() in MISSING_TOKENS

def normalize_text(v):
    if is_missing(v):
        return None
    return str(v).strip()

def parse_date(v):
    s = normalize_text(v)
    if s is None or s.lower() == 'still admitted':
        return None
    for fmt in DATE_FORMATS:
        try:
            return datetime.strptime(s, fmt).date()
        except ValueError:
            continue
    return None

def parse_bp(v):
    s = normalize_text(v)
    if s is None:
        return None, None
    m = re.match(r'^\s*(\d{2,3})\s*(?:/|over)\s*(\d{2,3})\s*$', s, re.IGNORECASE)
    if not m:
        return None, None
    return int(m.group(1)), int(m.group(2))

def clean_age(v):
    s = normalize_text(v)
    if s is None:
        return None
    try:
        a = int(float(s))
    except ValueError:
        return None
    if a < 0 or a > 120:
        return None
    return a

def standardize_weight(value, hint):
    """Return weight in kg (float) or None."""
    s = normalize_text(value)
    if s is None:
        return None
    try:
        w = float(s)
    except ValueError:
        return None
    h = normalize_text(hint)
    if h and h.lower() == 'lbs':
        return round(w / 2.20462, 2)
    return round(w, 2)

def mapped_lower(v, mapping, default=None):
    s = normalize_text(v)
    if s is None:
        return None
    return mapping.get(s.lower(), default)

In [ ]:
def clean_dataframe(df_in):
    df = df_in.copy()

    # Drop exact duplicates by patient_id
    n_before = len(df)
    df = df.drop_duplicates(subset=['patient_id']).reset_index(drop=True)
    print(f'Dropped {n_before - len(df)} duplicate rows.')

    # Demographics
    df['age']       = df['age'].map(clean_age)
    df['sex']       = df['sex'].map(lambda v: mapped_lower(v, SEX_MAP))
    df['race']      = df['race'].map(lambda v: mapped_lower(v, RACE_MAP, default='Other') if normalize_text(v) else None)
    df['insurance'] = df['insurance'].map(lambda v: mapped_lower(v, INSURANCE_MAP))
    df['smoking_status'] = df['smoking_status'].map(lambda v: mapped_lower(v, SMOKING_MAP))

    # Height — numeric only
    def to_float(v):
        s = normalize_text(v)
        if s is None:
            return None
        try:
            return float(s)
        except ValueError:
            return None
    df['height_cm'] = df['height_cm'].map(to_float)

    # Weight — convert lbs to kg using hint
    df['weight_kg'] = df.apply(lambda r: standardize_weight(r['weight'], r['weight_unit_hint']), axis=1)
    df = df.drop(columns=['weight', 'weight_unit_hint'])

    # Dates → ISO
    admit_parsed     = df['admission_date'].map(parse_date)
    discharge_parsed = df['discharge_date'].map(parse_date)

    # Length of stay: prefer derived from dates, else fall back to column
    def los_for(admit_d, disch_d, los_raw):
        if admit_d and disch_d:
            return (disch_d - admit_d).days
        s = normalize_text(los_raw)
        if s is None:
            return None
        try:
            return int(float(s))
        except ValueError:
            return None
    df['length_of_stay_days'] = [
        los_for(a, d, r) for a, d, r in zip(admit_parsed, discharge_parsed, df['length_of_stay_days'])
    ]
    df['admission_date'] = admit_parsed.map(lambda d: d.isoformat() if d else None)
    df['discharge_date'] = discharge_parsed.map(lambda d: d.isoformat() if d else None)

    # Lab test consolidation — must run BEFORE generic missingness so 'na' = Sodium
    def map_lab(v):
        if v is None or (isinstance(v, float) and pd.isna(v)):
            return None
        return LAB_TEST_MAP.get(str(v).strip().lower())
    df['lab_test_name'] = df['lab_test_name'].map(map_lab)

    # Lab value + unit standardization
    cleaned_vals, cleaned_units = [], []
    for test, raw_val, raw_unit in zip(df['lab_test_name'], df['lab_value'], df['lab_unit']):
        v, u = convert_lab(test, normalize_text(raw_val), normalize_text(raw_unit))
        cleaned_vals.append(v)
        cleaned_units.append(u)
    df['lab_value'] = cleaned_vals
    df['lab_unit']  = cleaned_units

    # Blood pressure → systolic / diastolic columns
    bp_parsed = df['blood_pressure'].map(parse_bp)
    df['bp_systolic']  = bp_parsed.map(lambda t: t[0])
    df['bp_diastolic'] = bp_parsed.map(lambda t: t[1])
    df = df.drop(columns=['blood_pressure'])

    # Diagnosis description: trim whitespace
    df['primary_diagnosis_desc'] = df['primary_diagnosis_desc'].map(normalize_text)

    return df

df_clean = clean_dataframe(df_raw)
print(f'\nCleaned shape: {df_clean.shape}')
df_clean.head(3)

In [ ]:
clean_csv = OUTPUT_DIR / 'patients_clean.csv'
df_clean.to_csv(clean_csv, index=False)
print(f'Wrote cleaned CSV → {clean_csv}  ({len(df_clean)} rows)')

# Quick sanity check of what got cleaned
print('\nSex values:',       df_clean['sex'].value_counts(dropna=False).to_dict())
print('Race values:',         df_clean['race'].value_counts(dropna=False).to_dict())
print('Smoking values:',      df_clean['smoking_status'].value_counts(dropna=False).to_dict())
print('Lab test values:',     df_clean['lab_test_name'].value_counts(dropna=False).to_dict())
print('Lab unit values:',     df_clean['lab_unit'].value_counts(dropna=False).to_dict())

## 4. Part B — Summarize each patient's notes with Ollama

Each chart at `/data/patient_notes/PT*.txt` is a multi-page text record (ED triage, ED MD note, H&P, daily progress notes, nursing notes, consults, imaging, discharge). We ask the LLM to produce a structured summary suitable for joining back to the cleaned CSV.

In [ ]:
def load_chart(pid):
    path = NOTES_DIR / f'{pid}.txt'
    if not path.exists():
        return None
    return path.read_text(encoding='utf-8')

def call_ollama(prompt, model=OLLAMA_MODEL, expect_json=True):
    """Call Ollama. When expect_json=True, force the runtime into JSON mode
    and lower temperature for stable structured output."""
    kwargs = {'model': model, 'prompt': prompt,
              'options': {'temperature': 0.1}}
    if expect_json:
        kwargs['format'] = 'json'
    response = ollama.generate(**kwargs)
    return response['response']

def safe_parse_json(text):
    if not text or not text.strip():
        raise ValueError('Model output is empty.')
    cleaned = text.strip()
    try:
        return json.loads(cleaned)
    except json.JSONDecodeError:
        pass
    m = re.search(r'```json\s*(\{.*?\})\s*```', cleaned, re.DOTALL)
    if m:
        return json.loads(m.group(1))
    m = re.search(r'(\{.*\})', cleaned, re.DOTALL)
    if m:
        return json.loads(m.group(1))
    raise ValueError('Could not find valid JSON in model output.')

In [ ]:
summary_task_statement = (
    'Read the patient chart text below and produce a concise structured summary '
    'suitable for a research dataset. Use only information explicitly present '
    'in the chart. If a field cannot be determined, write "not stated".'
)

summary_guardrail = (
    'Do not invent diagnoses, medications, or outcomes. Quote sparingly; '
    'summarize in plain English. Use null for any field you cannot determine. '
    'Return ONLY a single JSON object. No prose, no markdown fences.'
)

summary_schema = {
    'chief_complaint': 'one short phrase',
    'primary_diagnosis': 'free-text diagnosis label',
    'key_problems': ['short problem labels, may be empty'],
    'hospital_course_summary': '1-3 sentence narrative',
    'discharge_disposition': 'where the patient went on discharge',
    'length_of_stay_days': 'integer, or null if not stated',
}

summary_prompt_template = '''You are helping with a clinical research workflow on synthetic patient charts.

Task:
{task_statement}

Guardrail:
{guardrail}

Return the result as JSON only, with this structure:
{output_schema}

Patient chart:
{chart_text}
'''

print(summary_prompt_template[:400])

In [ ]:
# Test on a single chart first
pid_test = df_clean['patient_id'].iloc[0]
chart_text = load_chart(pid_test)
print(f'Patient: {pid_test} | chart length: {len(chart_text) if chart_text else 0} chars')

prompt = summary_prompt_template.format(
    task_statement=summary_task_statement,
    guardrail=summary_guardrail,
    output_schema=json.dumps(summary_schema, indent=2),
    chart_text=chart_text or '',
)
print('\nPrompt preview (first 800 chars):')
print(prompt[:800])

In [ ]:
# Run the LLM call against the test chart
raw_test = call_ollama(prompt)
print(raw_test)

In [ ]:
# Try parsing as JSON
parsed_test = safe_parse_json(raw_test)
print(json.dumps(parsed_test, indent=2))

In [ ]:
# Scale across a subset (LLM_SUBSET_N controls how many patients are processed).
# Bump up once your single-patient test looks reasonable.

summaries = []
target_ids = df_clean['patient_id'].head(LLM_SUBSET_N).tolist()

for i, pid in enumerate(target_ids, 1):
    chart_text = load_chart(pid)
    if chart_text is None:
        summaries.append({'patient_id': pid, 'error': 'chart not found', 'raw_output': None})
        continue
    prompt = summary_prompt_template.format(
        task_statement=summary_task_statement,
        guardrail=summary_guardrail,
        output_schema=json.dumps(summary_schema, indent=2),
        chart_text=chart_text,
    )
    raw = None
    try:
        raw = call_ollama(prompt)
        parsed = safe_parse_json(raw)
        summaries.append({'patient_id': pid, 'raw_output': raw, 'parsed': parsed})
    except Exception as e:
        summaries.append({'patient_id': pid, 'error': str(e), 'raw_output': raw})
    if i % 5 == 0:
        print(f'  {i}/{len(target_ids)} summaries done')

print(f'\nDone. Generated {len(summaries)} summaries.')

In [ ]:
summaries_path = OUTPUT_DIR / 'note_summaries.json'
with open(summaries_path, 'w', encoding='utf-8') as f:
    json.dump(summaries, f, indent=2)
print('Saved summaries →', summaries_path)

# Peek at one parsed summary
for s in summaries:
    if 'parsed' in s:
        print('\nExample parsed summary:')
        print(json.dumps(s['parsed'], indent=2))
        break

## 5. Part C — Extract a smoking flag from the notes

Workshop hook: the narrative notes were generated **independently** of the CSV `smoking_status` column, so the LLM-derived flag will sometimes agree, sometimes disagree, and sometimes be silent. Comparing the two is the point.

In [ ]:
smoking_task_statement = (
    'Determine whether the patient chart below indicates tobacco use. Look in '
    'the social history, history of present illness, physical exam, and '
    'discharge instructions. Subtle cues count (e.g., nicotine staining on '
    'fingers, "declines to specify", offered cessation referral, nicotine '
    'patch ordered) but do not invent evidence not present in the text.'
)

smoking_guardrail = (
    'If the chart says nothing about tobacco, return "Unknown". Do not infer '
    'tobacco use from a diagnosis alone (e.g., do not call a COPD patient a '
    'smoker unless the chart says so). Return ONLY a single JSON object. '
    'No prose, no markdown fences.'
)

smoking_schema = {
    'smoking_status_from_notes': 'one of: "Never", "Former", "Current", "Unknown"',
    'confidence': 'one of: "high", "medium", "low"',
    'evidence': ['short quotes or paraphrases from the chart, may be empty'],
}

smoking_prompt_template = '''You are reviewing a synthetic patient chart for evidence of tobacco use.

Task:
{task_statement}

Guardrail:
{guardrail}

Return JSON only, with this structure:
{output_schema}

Patient chart:
{chart_text}
'''

print(smoking_prompt_template[:400])

In [ ]:
# Test on one patient
pid_test = df_clean['patient_id'].iloc[0]
chart_text = load_chart(pid_test)
prompt = smoking_prompt_template.format(
    task_statement=smoking_task_statement,
    guardrail=smoking_guardrail,
    output_schema=json.dumps(smoking_schema, indent=2),
    chart_text=chart_text,
)
raw = call_ollama(prompt)
print('--- raw output ---')
print(raw)
print('\n--- parsed ---')
print(json.dumps(safe_parse_json(raw), indent=2))

In [ ]:
# Scale across the same subset
smoking_flags = []
for i, pid in enumerate(target_ids, 1):
    chart_text = load_chart(pid)
    if chart_text is None:
        smoking_flags.append({'patient_id': pid, 'error': 'chart not found'})
        continue
    prompt = smoking_prompt_template.format(
        task_statement=smoking_task_statement,
        guardrail=smoking_guardrail,
        output_schema=json.dumps(smoking_schema, indent=2),
        chart_text=chart_text,
    )
    raw = None
    try:
        raw = call_ollama(prompt)
        parsed = safe_parse_json(raw)
        smoking_flags.append({
            'patient_id': pid,
            'smoking_from_notes': parsed.get('smoking_status_from_notes'),
            'confidence': parsed.get('confidence'),
            'evidence': parsed.get('evidence'),
            'raw_output': raw,
        })
    except Exception as e:
        smoking_flags.append({'patient_id': pid, 'error': str(e), 'raw_output': raw})
    if i % 5 == 0:
        print(f'  {i}/{len(target_ids)} smoking flags done')

print(f'\nDone. Extracted flags for {len(smoking_flags)} patients.')

In [ ]:
# Merge note-derived smoking flag into cleaned dataframe
flags_df = pd.DataFrame([
    {'patient_id': f['patient_id'],
     'smoking_from_notes': f.get('smoking_from_notes'),
     'smoking_from_notes_confidence': f.get('confidence')}
    for f in smoking_flags
])

df_final = df_clean.merge(flags_df, on='patient_id', how='left')

# Cross-tab of CSV smoking_status vs note-derived smoking flag
comparison = (df_final
              .dropna(subset=['smoking_from_notes'])
              .groupby(['smoking_status', 'smoking_from_notes'], dropna=False)
              .size()
              .unstack(fill_value=0))
print('CSV (rows) vs Notes (cols):')
print(comparison)

In [ ]:
final_csv = OUTPUT_DIR / 'patients_final.csv'
df_final.to_csv(final_csv, index=False)

flags_path = OUTPUT_DIR / 'smoking_flags.json'
with open(flags_path, 'w', encoding='utf-8') as f:
    json.dump(smoking_flags, f, indent=2)

print('Saved final CSV   →', final_csv)
print('Saved flags JSON  →', flags_path)

## 6. Quick evaluation notes

Before scaling beyond the subset, reflect:

- **Summaries** — Are the extracted fields faithful to the chart? Where does the model hallucinate or repeat boilerplate?
- **Smoking flag** — How often does the note-derived flag disagree with the CSV `smoking_status`? Of the disagreements, which source is more trustworthy in each case? How does the model handle charts that are silent on tobacco?
- **Prompts** — Would tightening the schema or guardrail change behavior? What would you tweak?
- **Throughput** — How long does a single call take on this HPC node? Is `LLM_SUBSET_N` a useful ceiling to keep while iterating?

In [ ]:
# Save run metadata
timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
metadata = {
    'run_time': datetime.now().isoformat(),
    'model': OLLAMA_MODEL,
    'prompt_version': PROMPT_VERSION,
    'csv_path': str(CSV_PATH),
    'notes_dir': str(NOTES_DIR),
    'records_cleaned': len(df_clean),
    'records_summarized': len(summaries),
    'records_smoking_flagged': len(smoking_flags),
    'llm_subset_n': LLM_SUBSET_N,
    'output_files': {
        'cleaned_csv': str(clean_csv),
        'summaries_json': str(summaries_path),
        'smoking_flags_json': str(flags_path),
        'final_csv': str(final_csv),
    },
    'task_statements': {
        'summary': summary_task_statement,
        'smoking': smoking_task_statement,
    },
}
metadata_path = OUTPUT_DIR / f'metadata_{timestamp}.json'
with open(metadata_path, 'w', encoding='utf-8') as f:
    json.dump(metadata, f, indent=2)

print('Saved metadata →', metadata_path)
print(json.dumps(metadata, indent=2))

## 7. Optional next steps

- Bump `LLM_SUBSET_N` and re-run Parts B and C over the full 1000 patients.
- Tighten or relax the summary/smoking prompts; rerun and compare.
- Add a confusion matrix or simple agreement metric for CSV vs. note-derived smoking.
- Spot-check disagreements: open the chart `.txt` and decide which source you'd trust.
- Convert this notebook to a script and submit with `sbatch` for batch runs.